In [ ]:
# Import the clean_sac_file function from the previous example
from obspy import read, Trace, Stream
from obspy.signal.filter import bandpass
import numpy as np

def clean_sac_file(input_file, output_file, freq_min = 2.0, freq_max = 4.3, time_pad = 10.001):
    """
    Applies a bandpass filter to a SAC file to clean noise.
    Inputs:
    - input_file: path to the input SAC file
    - output_file: path to the output SAC file
    - freq_min: minimum frequency of the passband in Hz
    - freq_max: maximum frequency of the passband in Hz
    """
    # Read in the SAC file as an ObsPy Trace object
    trace = read(input_file)[0]
    # trace.plot()
    
    # Apply a bandpass filter to the trace
    trace_filtered = trace.copy()
    t = trace_filtered.stats.starttime

    if t + time_pad > trace_filtered.stats.endtime:
        return None
    
    trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime) 
    # trace_filtered = trace_filtered.trim(t, trace_filtered.stats.endtime)
    # trace_filtered.data = np.concatenate((np.zeros(750), trace_filtered.data))
    
    trace_filtered.data = bandpass(trace_filtered.data, freqmin=freq_min, freqmax=freq_max, df=trace_filtered.stats.sampling_rate)
    # trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime)

    max_val = max(trace_filtered.data[300:])

    if max_val > 4000:
        trace.plot()
        trace_filtered.plot()

    return trace_filtered

In [ ]:
import os
import shutil
from obspy import read, Trace, Stream

os.makedirs("class0") # yok
os.makedirs("class1") # 1 seviye 250
os.makedirs("class2") # 2 seviye 500
os.makedirs("class3") # 3 seviye 1000
os.makedirs("class4") # 4 seviye 2000
os.makedirs("class5") # 5 seviye 4000
os.makedirs("class6") # 6 seviye 8000   
os.makedirs("class7") # 6 seviye 16000   
os.makedirs("class8") # 6 seviye 32000   


# Define the directory path
dir_path = './'
orderdata = '202208_'

c = 0

clean_files = []

for i, folder_month in enumerate(os.listdir(dir_path)):

    if not os.path.isfile(dir_path + folder_month):
        # print(folder_month)

        for j, eartq in enumerate(os.listdir(dir_path + folder_month)):

            if not os.path.join(dir_path, folder_month, eartq).format().__contains__("BHZ"):
                continue

            # print(f"    {eartq}")
            
            date = str(i) + "_" + str(j) + "_" + str(folder_month)[-3:] + "_"
            
            # c = c + 1
            # continue

            # Define the input and output file paths
            input_file = dir_path + folder_month + '/' + eartq
            output_file = orderdata + date + eartq.replace('(', '').replace(')', '').replace('=', '').replace('KO', 'SAC')


            file_name = dir_path + folder_month + '/' + eartq


            if clean_files.__contains__(file_name):
                continue

            trace_filtered = clean_sac_file(input_file, output_file)
            if trace_filtered is None or len(trace_filtered.data) < 4000:
                continue

            clean_files.append(file_name)

            # if trace_filtered.stast.endtime - trace_filtered.stats.starttime < 15:
            #     continue

            min_val = min(trace_filtered.data[150:])
            max_val = max(trace_filtered.data[300:])

            # if max_val > 4000:
            #     trace_filtered.plot()

            max_value_index = np.argmax(trace_filtered.data[300:])
            # get the 1000 padding centered max index
            trace_filtered.data = trace_filtered.data[max_value_index - 2000: max_value_index + 2000]
           
            # print(f"min: {min_val} max: {max_val}")
            # print(file_name)
            # if max_val > 16000:
            #     trace_filtered.write('./class6/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class6/' + eartq + '_original.sac')

            if max_val < 250:
                trace_filtered.write('./class0/' + output_file, format='SAC')
                # shutil.copy(file_name, './class0/' + eartq + '_original.sac')
            elif max_val < 500:
                trace_filtered.write('./class1/' + output_file, format='SAC')
                # shutil.copy(file_name, './class1/' + eartq + '_original.sac')
            elif max_val < 1000:
                trace_filtered.write('./class2/' + output_file, format='SAC')
                # shutil.copy(file_name, './class2/' + eartq + '_original.sac')
            elif max_val < 2000:
                trace_filtered.write('./class3/' + output_file, format='SAC')
                # shutil.copy(file_name, './class3/' + eartq + '_original.sac')
            elif max_val < 4000:
                trace_filtered.write('./class4/' + output_file, format='SAC')
                # shutil.copy(file_name, './class4/' + eartq + '_original.sac')
            elif max_val < 8000:
                trace_filtered.write('./class5/' + output_file, format='SAC')
                # shutil.copy(file_name, './class5/' + eartq + '_original.sac')
            elif max_val < 16000:
                trace_filtered.write('./class6/' + output_file, format='SAC')
                # shutil.copy(file_name, './class6/' + eartq + '_original.sac')
            elif max_val < 32000:
                trace_filtered.write('./class7/' + output_file, format='SAC')
                # shutil.copy(file_name, './class7/' + eartq + '_original.sac')
            else:
                trace_filtered.write('./class8/' + output_file, format='SAC')
                # shutil.copy(file_name, './class8/' + eartq + '_original.sac')



            # if max_val < 500:
            #     trace_filtered.write('./class0/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class0/' + eartq + '_original.sac')
            # elif max_val < 1000:
            #     trace_filtered.write('./class1/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class1/' + eartq + '_original.sac')
            # elif max_val < 2000:
            #     trace_filtered.write('./class2/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class2/' + eartq + '_original.sac')
            # elif max_val < 4000:
            #     trace_filtered.write('./class3/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class3/' + eartq + '_original.sac')
            # elif max_val < 8000:
            #     trace_filtered.write('./class4/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class4/' + eartq + '_original.sac')
            # elif max_val < 16000:
            #     trace_filtered.write('./class5/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class5/' + eartq + '_original.sac')
            # else:
            #     trace_filtered.write('./class6/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class6/' + eartq + '_original.sac')

print(c)

In [1]:

#  read sac file

from obspy import read, Trace, Stream
import os

path = "./"
folder_classes = ["class8", "class7", "class6", "class5", "class4", "class3", "class2", "class1", "class0"]

for folder_class in folder_classes:
    files = []
    for j, file in enumerate(os.listdir(path= path + folder_class)):
        files.append(file)

    files = sorted(files)

    for i, file in enumerate(files):
        print(file)
        trace = read(path + folder_class + "/" + file)[0]
        print(len(trace.data))

        if len(trace.data) == 4000:
            print(trace.stats)
            # trace.plot()
            # ix = input("Press Enter to continue...")
            # if ix == 'q':
            #     break
        else:
            os.remove(path + folder_class + "/" + file)


/usr/lib/python3/dist-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 1.1build1 is an invalid version and will not be supported in a future release
  warnings.warn(
/usr/lib/python3/dist-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.43ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(


201901_10_0_SIMA.BHZ.SAC
4000
         network: KO
         station: SIMA
        location: --
         channel: BHZ
       starttime: 2019-01-27T01:37:05.000000Z
         endtime: 2019-01-27T01:37:44.990000Z
   sampling_rate: 100.0
           delta: 0.01
            npts: 4000
           calib: 1.0
         _format: SAC
             sac: AttribDict({'delta': 0.01, 'depmin': -33857.098, 'depmax': 37034.074, 'b': 10.0, 'e': 49.989998, 'o': 5.0, 'a': 116.746, 'internal0': 2.0, 't0': 118.369, 't1': 1.6230011, 'f': 0.0, 'stla': 39.0833, 'stlo': 28.9825, 'stel': 984.0, 'evla': 39.15517, 'evlo': 29.04717, 'evdp': 5.3, 'dist': 9.743986, 'az': 215.05055, 'baz': 35.009945, 'gcarc': 0.08764486, 'depmen': -0.13884053, 'cmpaz': 0.0, 'cmpinc': 0.0, 'nzyear': 2019, 'nzjday': 27, 'nzhour': 1, 'nzmin': 36, 'nzsec': 55, 'nzmsec': 0, 'nvhdr': 6, 'npts': 4000, 'iftype': 1, 'idep': 5, 'iztype': 9, 'leven': 1, 'lpspol': 1, 'lovrok': 1, 'lcalda': 1, 'unused23': 1, 'kstnm': 'SIMA', 'kevnm': '20190127013754',

In [ ]:


from obspy import read, Trace, Stream
import os

path = "./"

folder_class = "class8"

files = []
for j, file in enumerate(os.listdir(path= path + folder_class)):
    files.append(file)

files = sorted(files)

for i, file in enumerate(files):
    print(file)
    trace = read(path + folder_class + "/" + file)[0]
    print(len(trace.data))

    if len(trace.data) == 4000:
        # print(trace.stats)
        trace.plot()
        # ix = input("Press Enter to continue...")
        # if ix == 'q':
        #     break
        
    else:
        print(len(trace.data))
        # os.remove(path + folder_class + "/" + file)

folder_class = "class7"


files = []
for j, file in enumerate(os.listdir(path= path + folder_class)):
    files.append(file)

files = sorted(files)

for i, file in enumerate(files):
    print(file)
    trace = read(path + folder_class + "/" + file)[0]
    print(len(trace.data))

    if len(trace.data) == 4000:
        # print(trace.stats)
        trace.plot()
        # ix = input("Press Enter to continue...")
        # if ix == 'q':
        #     break
        
    else:
        print(len(trace.data))
        # os.remove(path + folder_class + "/" + file)


In [ ]:
from obspy import read, Trace, Stream

# path = "/home/mbulucay/Desktop/GradProject/2021/202103/20210311_07171939.16-29.05Ke-AHLATLICESME-SIMAV-KUTAHYA.M=1.6/SIMA.BHZ.KO"

path = "./class5/202204|_6_2_1.8_SIMA.BHZ.SAC"

trace = read(path)[0]

print(trace.stats.endtime - trace.stats.starttime)
print(len(trace.data))
trace.plot()


In [ ]:
import os

# Define the directory path
dir_path = './'

for folder_month in os.listdir(dir_path):

    if not os.path.isfile(dir_path + folder_month):
        print(folder_month)

        for eartq in os.listdir(dir_path + folder_month):
            print(f"    {eartq}")
            
            for sac_file in os.listdir(dir_path + folder_month + '/' + eartq):
                print(f"        {sac_file}")
                # Define the input and output file paths
                input_file = dir_path + folder_month + '/' + eartq + '/' + sac_file
                output_file = dir_path + folder_month + '/' + eartq + '/' + sac_file[:-4] + '_filtered.sac'

                # Apply the clean_sac_file function
                trace_filtered = clean_sac_file(input_file, output_file)
                trace_filtered.plot()
                # trace_filtered.write(output_file, format='SAC')
            
                ix = input("Press Enter to continue...")

                if ix == 'q':
                    break




In [ ]:
import glob

directory = './'  # Replace with the actual directory path you want to iterate over

# Find all items (files and directories) in the directory
items = glob.glob(directory + '/*')

# Iterate over the items
for item in items:
    if os.path.isfile(item):  # Check if the item is a file
        print(f"File: {os.path.basename(item)}")
    elif os.path.isdir(item):  # Check if the item is a directory
        print(f"Directory: {os.path.basename(item)}")
    else:
        print(f"Other: {os.path.basename(item)}")


In [ ]:

# #  read sac file


# from obspy import read, Trace, Stream

# path = "./"
# folder_class = "class6"

# for j, file in enumerate(os.listdir(path= path + folder_class)):
#     print(file)
#     trace = read(path + folder_class + "/" + file)[0]
#     trace.plot()
#     ix = input("Press Enter to continue...")
#     if ix == 'q':
#         break



